## This code will extract our data from the videos

I utilize an LLM to extract out numbers from the provided videos, but before that I will split the videos frames per second. Store these frames in Amazon S3 and then pass it to Open Ai to extract out the number. Do note this approach is not fully accurate and requires scanning through the csv file.

In [1]:
import cv2
import os

def split_video_to_images(video_path, output_dir, interval=1):
    cap = cv2.VideoCapture(video_path)
    frame_rate = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(frame_rate * interval)  # Number of frames to skip for 1-second intervals
    frame_index = 0
    count = 0

    os.makedirs(output_dir, exist_ok=True)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_index % frame_interval == 0:
            frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
            output_path = os.path.join(output_dir, f"frame_{count:03d}.jpg")
            cv2.imwrite(output_path, frame)
            count += 1
        frame_index += 1

    cap.release()
    print(f"Extracted {count} images to {output_dir}")

# split_video_to_images('traditional_mtr_vid\idle\idle_exp3.mp4', 'output_dir', 1)

## To extract data from the videos

You must add your own OPEN AI key or else this cell will not work. This cell function is to essentially extract the text out of your image via using OPEN AI

In [ ]:
import openai
from pytesseract import pytesseract as PT 
import base64
import cv2
import os

#code wont work unless passed the open ai key

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def extract_text_from_image(path):
    openai.api_key = 'YOUR OPEN AI KEY'
    

    response = openai.chat.completions.create(
        model= 'gpt-4o-mini',
        messages=[
        {"role": "system", "content": """ 
        Your are an assistant bot that will take an image of a smart meter and output the exact number produced by the smart meter, the number must be in decimal
        and it is in the middle of the smart meter. Do not return anything else just a number, for example the output will be {45.5}, nothing else.
        """},
        {'role': 'user', 'content' : [
                {
                    "type": "text",
                    "text": "What is in this image?",
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": path,
                    },
                },
            ],}]
    )
    return response

# Extracting data from videos

This part of the code will do the whole process of splitting the video into frames and then storing these images in S3 and finally passing it to the LLM one by one to generate a CSV file. One main thing to note here is the videos are not stored inside git lab.

In [ ]:
import boto3
import re
import pandas as pd
import os
import cv2

split_video_to_images('YOUR VIDEO', 'output_dir', 1)

result = []
s3 = boto3.resource('s3')
region = "us-east-1"
for bucket in s3.buckets.all():
    print(bucket.name)

count = 0
for image_file in os.listdir('output_dir'):
    if count <=311:
        image_path = os.path.join('output_dir', image_file)
        bucket_name = "individualprojectbucket"
        with open(image_path, 'rb') as data:
            s3.Bucket('individualprojectbucket').put_object(Key=image_file, Body=data)
        
        url_back = f"output_dir/{image_file}"
        url = f"https://individualprojectbucket.s3.us-east-1.amazonaws.com/{image_file}"
        response = extract_text_from_image(url)
        result.append(response)
        count+=1
        
print(len(result))
final_result = []
for results in result:
    numbers = re.findall(r'\{([\d\.]+)\}', results.choices[0].message.content)
    final_result.append(float(numbers[0]))

second = 0
data = {'Second': range(1, len(final_result) + 1), 'Smart_meter_reading': final_result}
df = pd.DataFrame(data)

df.to_csv("csv_file_traditional/traditional_meter_cpu_stresser_exp.csv", index=False)
print("Success")

Extracted 317 images to output_dir
individualprojectbucket
textract-console-ap-southeast-2-713c0780-e2d7-4a4e-9b9d-b3b0468
312
Success
